# Practice 2 — Intel Image Classification / Local XPU
Chạy từ trên xuống, kernel **Lab2 (.venv)**. Source nằm trong `src/phases/`; notebook chỉ điều phối.
WF-001 đã có benchmark riêng trong `outputs/environment/main.executed.ipynb`. WF-002 dưới đây dùng dữ liệu thật.
Restart kernel sau khi sửa source. Không chạy training CLI song song notebook.


In [ ]:
from pathlib import Path
import sys, inspect, json
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'configs/experiments.json').exists(), 'Open notebook from D:/Lab2'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.phases import phase_02_data, phase_03_models, phase_04_train, phase_05_evaluate
from IPython.display import display, Markdown, Image
PHASES = {2: phase_02_data, 3: phase_03_models, 4: phase_04_train, 5: phase_05_evaluate}
print('Python:', sys.executable)
print('Workspace:', ROOT)
print((ROOT / 'configs/experiments.json').read_text(encoding='utf-8'))


## Chọn phạm vi
`START_PHASE = END_PHASE` để chạy một phase; chọn 2–5 để chạy hết. Prerequisite phải tồn tại.
Phase 02 kiểm tra manifest; phase 03 kiểm chứng GPU; phase 04 train/resume hoặc xác minh run hoàn tất; phase 05 đánh giá test hoặc đọc kết quả test đã lưu.


In [ ]:
START_PHASE, END_PHASE = 2, 5
assert 2 <= START_PHASE <= END_PHASE <= 5
for number in range(START_PHASE, END_PHASE + 1):
    module = PHASES[number]
    print(f'PHASE-{number:02d}: {inspect.getfile(module)}')
    print(inspect.getsource(module.run))


In [ ]:
phase_results = {}
for number in range(START_PHASE, END_PHASE + 1):
    print(f'Running PHASE-{number:02d}', flush=True)
    phase_results[number] = PHASES[number].run()
    print(f'PHASE-{number:02d} finished', flush=True)


In [ ]:
report = ROOT / 'outputs/report/report.md'
if report.exists() and 5 in phase_results:
    display(Markdown('\n'.join(line for line in report.read_text(encoding='utf-8').splitlines() if not line.startswith('!['))))
    display(Image(filename=str(ROOT / 'outputs/report/learning_curves.png')))
    for run_id in phase_results[5]['selection']['test_runs']:
        display(Image(filename=str(ROOT / f'outputs/report/{run_id}_confusion_matrix.png')))
else:
    print('Partial pipeline: report is not presented as newly verified.')


In [ ]:
print('TensorBoard (run in a separate PowerShell terminal at D:/Lab2):')
print(r'.\.venv\Scripts\python.exe -m tensorboard.main --logdir outputs/experiments --host 127.0.0.1 --port 6006')
print('http://127.0.0.1:6006')
